# LangChain: Evaluation

## Goal of this notebook

Building LLM applications is not enough — we also need to evaluate their performance.

Unlike traditional machine learning models, LLM outputs are generative and open-ended.  
This makes evaluation more challenging.

In this notebook we explore how to evaluate a question-answering system using LLM-based evaluation techniques.

We will:

1. Build a question-answering system over documents
2. Generate evaluation questions
3. Produce answers from the system
4. Use an LLM to evaluate the quality of those answers

### Evaluation Pipeline

The evaluation workflow used in this notebook looks like this:

Documents  
      ↓  
RAG QA System  
      ↓  
Generated Answer  
      ↓  
Evaluation LLM  
      ↓  
Score / Feedback

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
# Use current model names to avoid migration warnings in modern LangChain.
llm_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

## Environment Setup

We begin by importing the required libraries and loading the OpenAI API key.

This notebook uses:

- LangChain
- OpenAI chat models
- vector embeddings
- evaluation utilities

In [4]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.globals import set_debug
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [5]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

## Creating the Vector Store

Next, we convert the documents into embeddings.

Embeddings represent text as numerical vectors that capture semantic meaning.

These embeddings are stored in a vector database (FAISS), which allows us to perform fast similarity search.

When a user asks a question, the system retrieves the most relevant document chunks from this vector store.

In [6]:
embeddings = OpenAIEmbeddings(model=embedding_model)
vectorstore = FAISS.from_documents(data, embeddings)
retriever = vectorstore.as_retriever()

## Creating the Vector Store

Next, we convert document chunks into embeddings.

These embeddings are stored in a vector database that supports similarity search.

When a user asks a question, we will retrieve the most relevant chunks from this database.

In [9]:
def format_docs(docs):
    return "\n<<<<>>>>>\n".join(doc.page_content for doc in docs)


qa_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If the answer is not in the context, say you do not know.
Answer concisely.

Context:
{context}

Question: {question}
""")

llm = ChatOpenAI(temperature=0.0, model=llm_model)
qa = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Keep a run-style alias so the later teaching cells still read naturally.
run = qa.invoke

In [10]:
run("What is LangChain?")

'I do not know.'

In [11]:
data[10]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [12]:
data[11]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

## Creating Evaluation Examples

To evaluate the QA system we need a set of question–answer examples.

These examples can be written manually or generated automatically using an LLM.

In this notebook we start with a few manually written examples and then generate additional examples from the documents.

In [13]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

In [14]:
example_gen_prompt = ChatPromptTemplate.from_template("""You are creating a short evaluation example from a product catalog entry.

Document:
{doc}

Generate one question that can be answered directly from the document, and provide the correct short answer.
Return valid JSON with exactly these keys:
- query
- answer
""")

In [15]:
example_gen_chain = (
    example_gen_prompt
    | ChatOpenAI(model=llm_model, temperature=0.0)
    | JsonOutputParser()
)

In [16]:
new_examples = [
    example_gen_chain.invoke({"doc": doc.page_content})
    for doc in data[:5]
]

In [17]:
new_examples[0]

{'query': "What material is used for the construction of the Women's Campside Oxfords?",
 'answer': 'Soft canvas material'}

In [18]:
data[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

### Combining Generated Examples

We combine the manually written evaluation examples with the newly generated ones.

This creates a larger evaluation dataset that will be used to test the QA system.

Having a diverse set of questions helps us better evaluate how well the system retrieves information and generates answers.

In [19]:
examples += new_examples

In [21]:
qa.invoke(examples[0]["query"])

'Yes, the Cozy Comfort Pullover Set has side pockets in the pull-on pants.'

## Building the Question-Answering System

We now create a Retrieval-Augmented Generation (RAG) pipeline.

The system works as follows:

1. Convert the user question into an embedding
2. Retrieve relevant document chunks
3. Provide those chunks as context to the language model
4. Generate an answer

In [22]:
set_debug(True)

In [23]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [

'Yes, the Cozy Comfort Pullover Set has side pockets in the pull-on pants.'

In [24]:
# Turn off the debug mode
set_debug(False)

## LLM-Based Evaluation

Evaluating LLM outputs manually can be time-consuming and subjective.

Instead, we can use another language model as a judge.

The evaluation model compares:

- the question
- the generated answer
- the reference answer

Based on this comparison, the model determines whether the answer is correct or incorrect.

This approach is commonly called **LLM-as-a-judge evaluation**.

In [25]:
predictions = [
    {
        **example,
        "result": qa.invoke(example["query"]),
    }
    for example in examples
]

In [26]:
eval_prompt = ChatPromptTemplate.from_template("""You are grading a question-answering system.

Question:
{query}

Reference answer:
{answer}

Predicted answer:
{result}

First, briefly explain whether the prediction matches the reference answer.
Then on a new final line write exactly one of:
CORRECT
INCORRECT
""")

In [27]:
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = eval_prompt | llm | StrOutputParser()

In [28]:
graded_outputs = [
    {"text": eval_chain.invoke(prediction)}
    for prediction in predictions
]

## Inspecting Evaluation Results

Finally, we review the evaluation results.

These results help us understand:

- which questions were answered correctly
- where the system made mistakes
- how retrieval quality affects answer accuracy

In [29]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set has side pockets in the pull-on pants.
Predicted Grade: The predicted answer matches the reference answer in that both confirm the presence of side pockets in the Cozy Comfort Pullover Set. However, the predicted answer provides additional detail about the location of the pockets, which is not present in the reference answer. Despite this extra information, the core response remains accurate.

CORRECT

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: The predicted answer matches the reference answer in that it correctly identifies the collection as the DownTek collection. The phrasing is slightly different, but the essential information is 

In [30]:
graded_outputs[0]

{'text': 'The predicted answer matches the reference answer in that both confirm the presence of side pockets in the Cozy Comfort Pullover Set. However, the predicted answer provides additional detail about the location of the pockets, which is not present in the reference answer. Despite this extra information, the core response remains accurate.\n\nCORRECT'}

## Conclusion

In this notebook we explored how to evaluate a question-answering system built with LangChain.

Key takeaways:

- LLM applications require systematic evaluation to measure quality.
- Evaluation datasets can be generated automatically using LLMs.
- LLM-based evaluation can assess answer correctness and relevance.
- Retrieval quality strongly affects the performance of RAG systems.

These evaluation techniques are essential when developing reliable AI systems for real-world applications.